# 02 · B2 Frozen sentence embeddings + linear SVC

Encodes texts with a frozen Indonesian sentence-embedding model (default: LazarusNLP/all-indobert-base-v4, set in configs/experiment.yaml) and trains only a linear SVC, following the LazarusNLP evaluation protocol. Runtime: Colab T4, a few minutes.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda", "No GPU: encoding on CPU takes 30+ minutes. Runtime -> Change runtime type -> T4 GPU, then rerun."

Device: cuda Tesla T4


In [3]:
import json, time, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from mad_indo import baselines, stats
model_name = cfg["baselines"]["frozen_embeddings"]["model"]
enc = SentenceTransformer(model_name, device=DEVICE)
out = RESULTS / "baselines"; out.mkdir(exist_ok=True)
cache = RESULTS / "embeddings"; cache.mkdir(exist_ok=True)

def embed(task, split, df):
    """Encode once and cache to Drive, so re-runs skip the slow step."""
    f = cache / f"{model_name.replace('/', '__')}_{task}_{split}_{len(df)}.npy"
    if f.exists():
        return np.load(f), 0.0
    t0 = time.perf_counter()
    E = enc.encode(list(df.text), batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    np.save(f, E)
    return E, time.perf_counter() - t0

summary = []
for task in ["smsa", "emot"]:
    L = cfg["tasks"][task]["labels"]
    tr, va = data.load_split(task, "train"), data.load_split(task, "valid")
    te = data.load_split(task, "test", limit=LIMIT)
    (Etr, s1), (Eva, s2), (Ete, s3) = embed(task, "train", tr), embed(task, "valid", va), embed(task, "test", te)
    print(task, "encoding seconds (0 = cached):", round(s1 + s2, 1), round(s3, 1))
    r = baselines.run_frozen(Etr, tr.label, Eva, va.label, Ete, L, cfg["baselines"]["frozen_embeddings"]["C_grid"])
    r.update({"encoder": model_name, "encode_train_valid_s": s1 + s2,
              "encode_infer_s_per_1000": 1000 * s3 / len(te) if s3 else None})
    pd.DataFrame({"id": te.id, "gold": te.label, "pred": r["pred"]}).to_csv(out / f"B2_frozen_{task}.csv", index=False)
    json.dump({k: v for k, v in r.items() if k != "pred"}, open(out / f"B2_frozen_{task}.json", "w"), indent=2)
    summary.append({"task": task, "method": "B2_frozen", "best_C": r["best_C"], **stats.metrics(te.label, r["pred"], L)})
    print(task, "done, best C =", r["best_C"])
pd.DataFrame(summary).round(4)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.37k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

... (12 outputs left out)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

emot encoding seconds (0 = cached): 14.5 1.7
emot done, best C = 0.1


,task,method,best_C,accuracy,macro_f1,unknown_rate,f1_positive,recall_positive,f1_neutral,recall_neutral,...,f1_sadness,recall_sadness,f1_anger,recall_anger,f1_love,recall_love,f1_fear,recall_fear,f1_happy,recall_happy
0,smsa,B2_frozen,5.0,0.8480,0.8009,0.0,0.8762,0.8846,0.6345,0.5227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,emot,B2_frozen,0.1,0.6364,0.6277,0.0,NaN,NaN,NaN,NaN,...,0.454,0.37,0.7352,0.8455,0.7481,0.7656,0.5785,0.5385,0.6226,0.6535
